In [ ]:
# ---- Install ONNX tooling (Internet must be ON) ----
!pip install -q onnx onnxruntime onnxscript


In [ ]:
# ---- Setup: find checkpoints, preprocess the test set once ----
import glob, json, os, time, zipfile
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, torch, timm, onnx, onnxruntime as ort
from PIL import Image
from sklearn.metrics import f1_score

IN, OUT = "/kaggle/input", "/kaggle/working"
ONNX_DIR = f"{OUT}/onnx"; os.makedirs(ONNX_DIR, exist_ok=True)
IMG = 224
N_THREADS = os.cpu_count()
print("onnxruntime", ort.__version__, "| torch", torch.__version__, "| CPU threads:", N_THREADS)

# Final checkpoints (MobileNet seed 1 = full-schedule rerun)
RUNS = {"mobilenetv3_small_100": ["0", "1_full", "2"], "mobilevit_xxs": ["0", "1", "2"]}
CKPT = {}
for model, seeds in RUNS.items():
    for s in seeds:
        hits = glob.glob(f"{IN}/**/{model}_final_seed{s}.pt", recursive=True)
        assert hits, f"{model}_final_seed{s}.pt not found - add the Phase 2 and rerun notebook outputs as inputs."
        CKPT[(model, s)] = hits[0]
        print(model, s, "->", hits[0])

def find_image_root(sample):
    if os.path.exists(os.path.join(IN, sample)):
        return IN
    first = sample.split("/")[0]
    for d in glob.glob(f"{IN}/**/{first}", recursive=True):
        if os.path.exists(os.path.join(os.path.dirname(d), sample)):
            return os.path.dirname(d)
    raise FileNotFoundError("Attach the PlantVillage dataset.")

df = pd.read_csv(glob.glob(f"{IN}/**/splits.csv", recursive=True)[0])
ROOT = find_image_root(df["path"].iloc[0])
CLASSES = df.drop_duplicates("label_idx").sort_values("label_idx")["label"].tolist()

def load_u8(paths):
    # Identical to Phase 2 eval transform before normalisation: PIL bilinear resize to 224x224
    def one(p):
        return np.asarray(Image.open(os.path.join(ROOT, p)).convert("RGB").resize((IMG, IMG), Image.BILINEAR))
    with ThreadPoolExecutor(N_THREADS) as ex:
        return np.stack(list(ex.map(one, paths)))

t0 = time.time()
test = df[df.split == "test"]
X_TEST, Y_TEST = load_u8(test["path"].tolist()), test["label_idx"].to_numpy()
calib = df[df.split == "train"].groupby("label_idx", group_keys=False).apply(
    lambda g: g.sample(min(len(g), 14), random_state=0))          # ~512 calibration images, all classes
X_CAL = load_u8(calib["path"].tolist())
print(f"test {X_TEST.shape}, calibration {X_CAL.shape}, loaded in {time.time() - t0:.0f}s")

def norm_stats(model):
    c = timm.models.get_pretrained_cfg(model).to_dict()
    return np.array(c["mean"], np.float32).reshape(1, 3, 1, 1), np.array(c["std"], np.float32).reshape(1, 3, 1, 1)

def to_input(u8, model):
    mean, std = norm_stats(model)
    return ((u8.astype(np.float32) / 255.0).transpose(0, 3, 1, 2) - mean) / std


In [ ]:
# ---- Diagnosis: is the INT8 collapse an x86 kernel problem (U8S8 saturation) or the models? ----
# ONNX Runtime docs: on x86 CPUs without VNNI, U8S8 kernels (uint8 activations x int8 weights) can saturate.
# Remedies: reduce_range=True (7-bit weights) or U8U8 (uint8 weights). ARM CPUs (Raspberry Pi) are not affected.
import subprocess
from onnxruntime.quantization import (CalibrationDataReader, QuantFormat, QuantType,
                                      quantize_dynamic, quantize_static)
from onnxruntime.quantization.shape_inference import quant_pre_process

flags = open("/proc/cpuinfo").read()
print("CPU:", subprocess.run("grep -m1 'model name' /proc/cpuinfo", shell=True, capture_output=True, text=True).stdout.strip())
print({f: (f in flags) for f in ["avx2", "avx512f", "avx512_vnni", "avx_vnni", "amx_int8"]})


class CalibReader(CalibrationDataReader):
    def __init__(self, x, bs=16):
        self.batches = iter([{"input": x[i:i + bs]} for i in range(0, len(x), bs)])

    def get_next(self):
        return next(self.batches, None)


def predict(path, x, bs=64):
    so = ort.SessionOptions(); so.intra_op_num_threads = N_THREADS
    sess = ort.InferenceSession(path, so, providers=["CPUExecutionProvider"])
    return np.concatenate([sess.run(None, {"input": x[i:i + bs]})[0].argmax(1) for i in range(0, len(x), bs)])


rng = np.random.default_rng(0)
idx = rng.choice(len(X_TEST), min(1000, len(X_TEST)), replace=False)   # 1,000 test images keep this quick
U8, S8 = QuantType.QUInt8, QuantType.QInt8
CONFIGS = {   # name: (kind, activation_type, weight_type, reduce_range)
    "static_U8S8 (as before)": ("static", U8, S8, False),
    "static_U8S8_reduce_range": ("static", U8, S8, True),
    "static_U8U8": ("static", U8, U8, False),
    "static_S8S8": ("static", S8, S8, False),
    "dynamic_S8 (as before)": ("dynamic", None, S8, False),
    "dynamic_S8_reduce_range": ("dynamic", None, S8, True),
    "dynamic_U8": ("dynamic", None, U8, False),
}
diag = []
for model_name, s in [("mobilenetv3_small_100", "0"), ("mobilevit_xxs", "0")]:
    tag = f"{model_name}_seed0"
    fp32 = f"{ONNX_DIR}/{tag}_fp32.onnx"
    if not os.path.exists(fp32):  # re-export if this session has no files from the previous run
        m = timm.create_model(model_name, pretrained=False, num_classes=len(CLASSES))
        m.load_state_dict(torch.load(CKPT[(model_name, s)], map_location="cpu")); m.eval()
        try:
            torch.onnx.export(m, torch.randn(1, 3, IMG, IMG), fp32, input_names=["input"], output_names=["logits"],
                              opset_version=17, dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}}, dynamo=False)
        except TypeError:
            torch.onnx.export(m, torch.randn(1, 3, IMG, IMG), fp32, input_names=["input"], output_names=["logits"],
                              opset_version=17, dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}})
    pre = f"{ONNX_DIR}/{tag}_pre.onnx"
    quant_pre_process(fp32, pre)
    x, y = to_input(X_TEST[idx], model_name), Y_TEST[idx]
    x_cal = to_input(X_CAL, model_name)
    ref = predict(fp32, x)
    print(f"\n===== {tag}: FP32 acc {np.mean(ref == y):.4f} on {len(y)} images =====", flush=True)
    for name, (kind, at, wt, rr) in CONFIGS.items():
        out = f"{ONNX_DIR}/{tag}_diag_{name.split(' ')[0]}.onnx"
        try:
            if kind == "static":
                quantize_static(pre, out, CalibReader(x_cal), quant_format=QuantFormat.QDQ, per_channel=True,
                                activation_type=at, weight_type=wt, reduce_range=rr)
            else:
                quantize_dynamic(pre, out, weight_type=wt, reduce_range=rr)
            p = predict(out, x)
            row = {"model": model_name, "config": name, "acc": float(np.mean(p == y)),
                   "agree_with_fp32": float(np.mean(p == ref)), "size_MB": os.path.getsize(out) / 2**20}
        except Exception as e:
            row = {"model": model_name, "config": name, "error": f"{type(e).__name__}: {e}"[:200]}
        diag.append(row)
        print(f"  {name:28s} " + (f"acc {row['acc']:.4f}  agree {row['agree_with_fp32']:.4f}"
                                  if "acc" in row else row["error"]), flush=True)
    os.remove(pre)

pd.DataFrame(diag).to_csv(f"{OUT}/int8_diagnosis.csv", index=False)
with zipfile.ZipFile(f"{OUT}/int8_diag_onnx.zip", "w", zipfile.ZIP_DEFLATED) as z:   # for testing on the Pi too
    for f in glob.glob(f"{ONNX_DIR}/*_diag_*.onnx"):
        z.write(f, f"onnx/{os.path.basename(f)}")
print("\nSaved int8_diagnosis.csv and int8_diag_onnx.zip")
